# MWIS QUBO, Annealing, and Reconstruction

This self-contained MWIS notebook reads the saved terminal subgraphs and decomposition map. It builds their QUBOs, runs D-Wave's classical simulated annealer, and combines the terminal choices and any explicit heuristic fallbacks into one schedule. It does not rebuild the conflict graph or decomposition.

This run is a classical simulation, not a quantum-hardware run. The saved QUBOs can be used separately with a real QPU when hardware access and credentials are available.

In [ ]:
from __future__ import annotations

import gzip
import json
import math
import os
from itertools import combinations
from pathlib import Path
from typing import Any

import networkx as nx
import pandas as pd

def find_project_root(start: Path | None = None) -> Path:
    """
    Find the repository root so paths work without further trouble.
    """
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "data").exists() and (candidate / "results").exists():
            return candidate
    raise FileNotFoundError("Could not find project root containing both `data/` and `results/`.")

# loading project root dir
PROJECT_ROOT = find_project_root()
RESULTS_DIR = PROJECT_ROOT / "results"


# --------- loading configuration environment ----------
# load .json config file
CONFIG_PATH = Path(os.environ.get("SWE2_CONFIG", RESULTS_DIR / "_config.json"))
run_config = json.loads(CONFIG_PATH.read_text(encoding="utf-8"))

# Bound the classical rescue work on separator-incomplete natural components.
MILP_PER_COMPONENT_SECONDS = float(run_config["milp_per_component_seconds"])
MILP_TOTAL_SOLVER_SECONDS = float(run_config["milp_total_solver_seconds"])
TRIANGLE_LP_PER_COMPONENT_SECONDS = float(run_config["triangle_lp_per_component_seconds"])
TRIANGLE_LP_TOTAL_SOLVER_SECONDS = float(run_config["triangle_lp_total_solver_seconds"])

# input: any stage that produced terminal subgraphs (MWIS decomposition, CSP showcase, ...)
INPUT_RUN_DIR = RESULTS_DIR / run_config["qubo_input_run"]
PRODUCER_CONFIG_PATH = INPUT_RUN_DIR / "run_config_log.json"
producer_config = json.loads(PRODUCER_CONFIG_PATH.read_text(encoding="utf-8")) if PRODUCER_CONFIG_PATH.exists() else {}
# load last stage final subgraphs
TERMINAL_SUBGRAPHS_JSON = INPUT_RUN_DIR / "terminal_subgraphs.json"
if not TERMINAL_SUBGRAPHS_JSON.exists():
    raise FileNotFoundError(f"No terminal_subgraphs.json in {INPUT_RUN_DIR}. Run the producing stage first.")
terminal_subgraphs = json.loads(TERMINAL_SUBGRAPHS_JSON.read_text(encoding="utf-8"))

# output: named after the input run, so QUBOs from different producers stay side by side
OUTPUT_RUN_NAME = run_config.get("qubo_output_run_name", INPUT_RUN_DIR.name)
# TODO: fix this for CSP run also
OUTPUT_RUN_DIR = RESULTS_DIR / "runs" / "mwis" / "qubo_annealing" / OUTPUT_RUN_NAME
OUTPUT_RUN_DIR.mkdir(parents=True, exist_ok=True)

# QUBO coefficients: scale the reward (linear) and the conflict penalty (quadratic).
# The penalty is computed from the effective reward scale, so changing the reward factor
# cannot accidentally make invalid conflicting selections attractive.
REWARD_FACTOR = float(run_config["qubo_reward_factor"])
PENALTY_FACTOR = float(run_config["qubo_penalty_factor"])
if not math.isfinite(REWARD_FACTOR) or REWARD_FACTOR <= 0:
    raise ValueError("qubo_reward_factor must be a positive finite number.")
if not math.isfinite(PENALTY_FACTOR) or PENALTY_FACTOR < 1.0:
    raise ValueError("qubo_penalty_factor must be a finite number >= 1.0 to preserve the MWIS penalty guarantee.")

# simulated-annealing settings
NUM_READS = run_config["num_reads"]
BASE_SEED = run_config["base_seed"]

# Log the selected producer settings with the annealing settings actually used.
run_config_log = {
    "producer_run": INPUT_RUN_DIR.name,
    "candidate_slot_csv": producer_config.get("candidate_slot_csv"),
    "max_qpu_terminal_nodes": producer_config.get("max_qpu_terminal_nodes"),
    "max_qpu_terminal_couplers": producer_config.get("max_qpu_terminal_couplers"),
    "exact_enumeration_node_limit": producer_config.get("exact_enumeration_node_limit"),
    "exact_clique_node_limit": producer_config.get("exact_clique_node_limit"),
    "qubo_input_run": run_config["qubo_input_run"],
    "qubo_reward_factor": REWARD_FACTOR,
    "qubo_penalty_factor": PENALTY_FACTOR,
    "num_reads": NUM_READS,
    "base_seed": BASE_SEED,
    "milp_per_component_seconds": MILP_PER_COMPONENT_SECONDS,
    "milp_total_solver_seconds": MILP_TOTAL_SOLVER_SECONDS,
    "triangle_lp_per_component_seconds": TRIANGLE_LP_PER_COMPONENT_SECONDS,
    "triangle_lp_total_solver_seconds": TRIANGLE_LP_TOTAL_SOLVER_SECONDS,
}
(OUTPUT_RUN_DIR / "run_config_log.json").write_text(json.dumps(run_config_log, indent=2), encoding="utf-8")


print(f"Loaded configuration from: {CONFIG_PATH.resolve()}")
print(f"Loaded terminal subgraphs from: {INPUT_RUN_DIR.resolve()}")
print(f"Artifacts will be saved to: {OUTPUT_RUN_DIR.resolve()}")
print({"reward_factor": REWARD_FACTOR, "penalty_factor": PENALTY_FACTOR, "penalty_rule": "penalty_factor * (max effective reward + 1)", "num_reads": NUM_READS})
print({"terminal_subgraphs": len(terminal_subgraphs)})

## Build terminal QUBOs

In [ ]:
def graph_from_terminal(record: dict[str, Any]) -> nx.Graph:
    """
    Rebuild one terminal subgraph from its JSON record.
    """
    graph = nx.Graph()
    # re-building nodes
    for node in record["graph"]["nodes"]:
        node = dict(node)
        node_id = node.pop("id")          # "id" is the key, the rest are attributes (weight, times, ...)
        graph.add_node(node_id, **node)
    # re-building edges 
    for edge in record["graph"]["edges"]:
        edge = dict(edge)
        source = edge.pop("source")
        target = edge.pop("target")
        graph.add_edge(source, target, **edge)   # keeps the conflict "reason" attribute
    return graph


def max_effective_reward(graph: nx.Graph) -> float:
    """
    Largest scaled node reward in the terminal graph given.
    """
    rewards = [REWARD_FACTOR * float(data["weight"]) for _, data in graph.nodes(data=True)]
    return max([0.0, *rewards])


def default_penalty(graph: nx.Graph) -> float:
    """
    Compute the conflict penalty, relative to the largest effective reward.
    """
    return PENALTY_FACTOR * (max_effective_reward(graph) + 1.0)


def build_qubo_from_graph(graph: nx.Graph, penalty: float | None = None) -> dict[str, Any]:
    """
    Build QUBO instance startng from the given graph.
    """
    if penalty is None:
        penalty = default_penalty(graph)

    # linear reward
    linear = {node: -REWARD_FACTOR * float(data["weight"]) for node, data in graph.nodes(data=True)}

    # quadratic penalty
    quadratic = {(u, v): float(penalty) for u, v in graph.edges}

    return {
        "variables": sorted(graph.nodes),
        "linear": linear,
        "quadratic": quadratic,
        "penalty": float(penalty),
        "max_effective_reward": max_effective_reward(graph),
    }


def qubo_energy(qubo: dict[str, Any], sample: dict[str, int]) -> float:
    """
    Compute the energy of the QUBO assignment.
    """
    energy = sum(coef * sample.get(node, 0) for node, coef in qubo["linear"].items())
    energy += sum(coef * sample.get(u, 0) * sample.get(v, 0) for (u, v), coef in qubo["quadratic"].items())
    return energy


qubo_records = []   # for each temrinal graph build a separate QUBO formulation
term_rows = []      # flat list of every QUBO coefficient
for record in terminal_subgraphs:
    graph = graph_from_terminal(record)
    qubo = build_qubo_from_graph(graph)
    qubo_records.append({"terminal": record, "graph": graph, "qubo": qubo})

    for node, coef in qubo["linear"].items():
        term_rows.append({"terminal_id": record["terminal_id"], "term_type": "linear", "u": node, "v": "", "coefficient": coef})
    for (u, v), coef in qubo["quadratic"].items():
        term_rows.append({"terminal_id": record["terminal_id"], "term_type": "quadratic", "u": u, "v": v, "coefficient": coef})

# print out translation summary
qubo_summary = pd.DataFrame([
    {
        "terminal_id": item["terminal"]["terminal_id"],
        "original_component_id": item["terminal"]["original_component_id"],
        "variables": item["graph"].number_of_nodes(),
        "couplers": item["graph"].number_of_edges(),
        "max_effective_reward": item["qubo"]["max_effective_reward"],
        "penalty": item["qubo"]["penalty"],
        "penalty_margin": item["qubo"]["penalty"] - item["qubo"]["max_effective_reward"],
    }
    for item in qubo_records
])
qubo_terms = pd.DataFrame(term_rows)

qubo_summary.to_csv(OUTPUT_RUN_DIR / "terminal_qubo_summary.csv", index=False)
qubo_terms.to_csv(OUTPUT_RUN_DIR / "terminal_qubo_terms.csv", index=False)
qubo_summary.head(10)

## Classical simulated annealing

In [ ]:
def selected_nodes_from_sample(sample: dict[str, int]) -> set[str]:
    """Extract the set of variables selected by the annealing sample given as input"""
    return {node for node, value in sample.items() if int(value) == 1}


def is_independent(graph: nx.Graph, selected: set[str]) -> bool:
    """Feasibility check: no two selected slots may conflict (penalty high enough)."""
    return all(not graph.has_edge(u, v) for u, v in combinations(selected, 2))


def sample_objective(graph: nx.Graph, selected: set[str]) -> float:
    return sum(float(graph.nodes[node]["weight"]) for node in selected)


def choose_best_feasible_read(response: Any, graph: nx.Graph, qubo: dict[str, Any]) -> dict[str, Any]:
    """Prefer the best feasible annealing read; fall back to the lowest-energy read if needed."""
    reads = []
    for row in response.data(["sample", "energy"]):
        sample = {variable: int(row.sample.get(variable, 0)) for variable in qubo["variables"]}
        selected = selected_nodes_from_sample(sample)
        reads.append({
            "sample": sample,
            "selected_nodes": selected,
            "objective": sample_objective(graph, selected),
            "energy": float(row.energy),
            "is_independent": is_independent(graph, selected),
        })

    feasible_reads = [read for read in reads if read["is_independent"]]
    if feasible_reads:
        return max(feasible_reads, key=lambda read: (read["objective"], -read["energy"], tuple(sorted(read["selected_nodes"]))))
    return min(reads, key=lambda read: (read["energy"], -read["objective"]))

# library check
try:
    from dwave.samplers import SimulatedAnnealingSampler
except ImportError as exc:
    raise ModuleNotFoundError(
        "D-Wave simulated annealing is required for this notebook. "
        "Install `dwave-samplers` in the Python 3.13.14 environment, "
        "then restart the kernel and rerun the notebook."
    ) from exc

annealing_rows = []
sampler = SimulatedAnnealingSampler()   # classical stand-in for the QPU, same interface of quantum annealer
for index, item in enumerate(qubo_records):
    qubo = item["qubo"]
    graph = item["graph"]
    terminal_id = item["terminal"]["terminal_id"]

    # The sampler wants one dict: diagonal keys (i, i) carry the linear terms.
    Q = {(node, node): coef for node, coef in qubo["linear"].items()}
    Q.update(qubo["quadratic"])

    # simulate quantum annealing
    response = sampler.sample_qubo(Q, num_reads=NUM_READS, seed=BASE_SEED + index)
    best = choose_best_feasible_read(response, graph, qubo)

    # convert the output into interpretable solution
    sample = best["sample"]
    selected = best["selected_nodes"]
    annealing_rows.append({
        "terminal_id": terminal_id,
        "selected_node_count": len(selected),
        "selected_nodes": json.dumps(sorted(selected)),
        "objective": best["objective"],
        "energy": best["energy"],
        "lowest_energy": float(response.first.energy),
        "recomputed_energy": qubo_energy(qubo, sample),
        "is_independent": best["is_independent"],
        "num_reads": NUM_READS,
        "seed": BASE_SEED + index,
    })

pd.DataFrame(annealing_rows).to_csv(OUTPUT_RUN_DIR / "annealing_results.csv", index=False)
print({"annealed_terminals": len(annealing_rows), "output": str(OUTPUT_RUN_DIR / "annealing_results.csv")})

## Bounded classical fallback for incomplete components

This helper is used after annealing, only for natural components whose separator decomposition was incomplete. It tries a time-limited binary MWIS solve with valid edge and triangle constraints. A proven optimum replaces that component’s contribution; otherwise, a validated incumbent and the best available upper bound are retained. The per-component and total time limits are configured above.

This is classical post-processing. It does not replace annealing for the other components.

In [ ]:
"""MILP fallback for separator-incomplete components."""

from __future__ import annotations

import time
from typing import Any
from pathlib import Path

import networkx as nx
import numpy as np
from scipy.optimize import Bounds, LinearConstraint, linprog, milp
from scipy.sparse import coo_matrix


TOLERANCE = 1e-6


def _edge_and_triangle_rows(
    graph: nx.Graph, node_index: dict[str, int]
) -> tuple[list[tuple[int, int]], list[tuple[int, int, int]]]:
    """
    Find all cliques of size 2 and size 3 within the conflict graph: size 2 cliques are used
    for conflicts, size 3 cliques are used to compute upper bounds.
    """
    edges = [(node_index[left], node_index[right]) for left, right in graph.edges]
    triangles: list[tuple[int, int, int]] = []
    for left in graph:
        for middle in graph[left]:
            if node_index[middle] <= node_index[left]:
                continue
            for right in graph[left].keys() & graph[middle].keys():
                if node_index[right] > node_index[middle]:
                    triangles.append((node_index[left], node_index[middle], node_index[right]))
    return edges, triangles


def _constraint_matrix(
    node_count: int,
    edges: list[tuple[int, int]],
    triangles: list[tuple[int, int, int]],
):
    """
    Starting from  edges (size 2 cliques) and triangles (size 3 cliques) build the 
    constraint matrix to be used in LP algorithms.
    """
    row_count = len(edges) + len(triangles)
    row_ids = np.empty(2 * len(edges) + 3 * len(triangles), dtype=np.int32)
    col_ids = np.empty_like(row_ids)
    cursor = 0
    for row, (left, right) in enumerate(edges):
        row_ids[cursor : cursor + 2] = row
        col_ids[cursor : cursor + 2] = (left, right)
        cursor += 2
    for row, triangle in enumerate(triangles, start=len(edges)):
        row_ids[cursor : cursor + 3] = row
        col_ids[cursor : cursor + 3] = triangle
        cursor += 3
    return coo_matrix(
        (np.ones(len(row_ids)), (row_ids, col_ids)),
        shape=(row_count, node_count),
    ).tocsc()


def triangle_lp_upper_bound(graph: nx.Graph, time_limit_seconds: float) -> dict[str, Any]:
    """
    Return an LP upper bound on the given component, or None if the bounded LP did not finish.
    """
    
    nodes = sorted(graph.nodes) # fix order just for repoducibility
    weights = np.array([float(graph.nodes[node]["weight"]) for node in nodes])
    if not nodes:
        return {"upper_bound": 0.0, "solve_seconds": 0.0, "status": "optimal"}
    
    index = {node: i for i, node in enumerate(nodes)}
    edges, triangles = _edge_and_triangle_rows(graph, index)
    matrix = _constraint_matrix(len(nodes), edges, triangles)
    constraints = (matrix, np.ones(len(edges) + len(triangles))) if matrix.shape[0] else None
   
    started = time.perf_counter()
    result = linprog(
        c=-weights,
        A_ub=constraints[0] if constraints else None,
        b_ub=constraints[1] if constraints else None,
        bounds=(0.0, 1.0),
        method="highs",
        options={"time_limit": float(time_limit_seconds)},
    )
    elapsed = time.perf_counter() - started
    
    return {
        "upper_bound": float(-result.fun) if result.status == 0 else None,
        "solve_seconds": elapsed,
        "status": "optimal" if result.status == 0 else str(result.message),
        "verified_triangles": len(triangles),
    }


def _solve_binary_mwis(graph: nx.Graph, time_limit_seconds: float) -> dict[str, Any]:
    """
    Solve problem instance with MILP.
    """
    
    # problem init
    nodes = sorted(graph.nodes)
    weights = np.array([float(graph.nodes[node]["weight"]) for node in nodes])
    index = {node: i for i, node in enumerate(nodes)}
    edges, triangles = _edge_and_triangle_rows(graph, index)
    matrix = _constraint_matrix(len(nodes), edges, triangles)
    constraints = LinearConstraint(matrix, -np.inf, np.ones(matrix.shape[0])) if matrix.shape[0] else None

    # problem solver (now sets the integrality bounds)
    started = time.perf_counter()
    result = milp(
        c=-weights,
        integrality=np.ones(len(nodes), dtype=np.uint8),
        bounds=Bounds(np.zeros(len(nodes)), np.ones(len(nodes))),
        constraints=constraints,
        options={"time_limit": float(time_limit_seconds), "mip_rel_gap": 0.0},
    )
    elapsed = time.perf_counter() - started

    # solver might force integrty by pushing assignment as very high (.999999) or 
    # very low (1e-8), but might not be exct integers. With this control we enforce integer values.
    selected = (
        {nodes[i] for i, value in enumerate(result.x) if value > 0.5}
        if result.x is not None
        else set()
    )

    
    feasible = bool(
        result.x is not None
        # TODO: consider dropping these
        # and np.all((result.x >= -TOLERANCE) & (result.x <= 1.0 + TOLERANCE))
        # and np.all(np.abs(result.x - np.round(result.x)) <= TOLERANCE)
        and all(not (left in selected and right in selected) for left, right in graph.edges)
    )

    # schedule value
    score = sum(float(graph.nodes[node]["weight"]) for node in selected) if feasible else None

    # get the bound from the MILP solver
    dual = getattr(result, "mip_dual_bound", None)
    upper_bound = float(-dual) if dual is not None and np.isfinite(dual) else None
    
    # if optimal (MILP) and feasible then proven to be optimal
    proven = bool(result.status == 0 and feasible)
    if proven:
        upper_bound = float(score)
    
    return {
        "status": str(result.message),
        "solver_status": int(result.status),
        "optimality_proved": proven,
        "selected_nodes": selected if feasible else set(),
        "objective": float(score) if score is not None else None,
        "upper_bound": upper_bound,
        "feasible": bool(feasible),
        "validation_passed": bool(
            feasible
            and (score is not None)
            and (upper_bound is None or upper_bound + TOLERANCE >= float(score))
        ),
        "solve_seconds": elapsed,
        "mip_nodes": getattr(result, "mip_node_count", None),
        "solver_relative_gap": getattr(result, "mip_gap", None),
        "verified_triangles": len(triangles),
        "component_nodes": len(nodes),
        "component_edges": len(edges),
    }


def solve_incomplete_component(
    graph: nx.Graph,
    current_selected: set[str],
    previous_upper_bound: float,
    milp_time_limit_seconds: float,
    lp_time_limit_seconds: float,
) -> dict[str, Any]:
    """
    Take one subproblem get from heuristic split, and solve it using MILP.

    An optimal MILP replaces the current component schedule. If the MILP times
    out, a valid better incumbent may improve the schedule; the triangle LP is
    then used to tighten the bound. The previous conservative bound remains a
    safe fallback if the LP does not finish.
    """
    started = time.perf_counter()
    current_score = sum(float(graph.nodes[node]["weight"]) for node in current_selected)
    if any(left in current_selected and right in current_selected for left, right in graph.edges):
        raise ValueError("Current component schedule is not conflict-free.")
    previous_upper_bound = float(previous_upper_bound)
    if previous_upper_bound + TOLERANCE < current_score:
        raise ValueError("Previous component upper bound is below its feasible schedule.")

    milp_result = None
    if milp_time_limit_seconds > 0:
        milp_result = _solve_binary_mwis(graph, milp_time_limit_seconds)

    lp_result = None
    if not milp_result or not milp_result["optimality_proved"]:
        if lp_time_limit_seconds > 0:
            lp_result = triangle_lp_upper_bound(graph, lp_time_limit_seconds)

    chosen = set(current_selected)
    chosen_score = current_score
    source = "annealing reconstruction"
    if milp_result and milp_result["feasible"]:
        milp_score = float(milp_result["objective"])
        if milp_result["optimality_proved"] and milp_score + TOLERANCE < current_score:
            raise ValueError("Proven MILP optimum is below a feasible input schedule.")
        if milp_result["optimality_proved"] or milp_score > chosen_score + TOLERANCE:
            chosen = set(milp_result["selected_nodes"])
            chosen_score = milp_score
            source = "MILP optimal" if milp_result["optimality_proved"] else "MILP feasible incumbent"

    if milp_result and milp_result["optimality_proved"]:
        upper_bound = chosen_score
        bound_source = "MILP optimality proof"
    else:
        bounds = [previous_upper_bound]
        bound_source = "previous conservative bound"
        if lp_result and lp_result["upper_bound"] is not None:
            bounds.append(float(lp_result["upper_bound"]))
            bound_source = "triangle-LP bound"
        if milp_result and milp_result["upper_bound"] is not None:
            dual_bound = float(milp_result["upper_bound"])
            if dual_bound + TOLERANCE >= chosen_score:
                bounds.append(dual_bound)
                if dual_bound <= min(bounds) + TOLERANCE:
                    bound_source = "MILP dual bound"
        upper_bound = min(bounds)
    if upper_bound + TOLERANCE < chosen_score:
        raise ValueError("Combined component upper bound is below its feasible objective.")

    return {
        "selected_nodes": chosen,
        "objective": chosen_score,
        "upper_bound": upper_bound,
        "optimality_proved": bool(milp_result and milp_result["optimality_proved"]),
        "solution_source": source,
        "bound_source": bound_source,
        "milp": milp_result,
        "triangle_lp": lp_result,
        "current_objective": current_score,
        "objective_improvement": chosen_score - current_score,
        "wall_seconds": time.perf_counter() - started,
    }


def apply_milp_fallback(
    run_dir: Path,
    component_records: list[dict[str, Any]],
    component_results: list[dict[str, Any]],
    component_rows: list[dict[str, Any]],
    total_upper_bound: float,
    milp_per_component_seconds: float = 5.0,
    milp_total_seconds: float = 30.0,
    lp_per_component_seconds: float = 5.0,
    lp_total_seconds: float = 30.0,
) -> dict[str, Any]:
    """
    Apply bounded MILP to incomplete natural components after annealing.

    The saved decomposition remains the source of which natural components
    are incomplete. All other reconstructed components are left untouched.
    """

    started = time.perf_counter()
    # load graphs
    node_table = pd.read_csv(run_dir / "conflict_graph_nodes.csv")
    edge_table = pd.read_csv(run_dir / "conflict_graph_edges.csv")
    graph = nx.Graph()
    graph.add_nodes_from(
        (str(row.slot_id), {"weight": float(row.weight)})
        for row in node_table.itertuples(index=False)
    )
    graph.add_edges_from(
        (str(left), str(right))
        for left, right in edge_table[["source", "target"]].itertuples(index=False, name=None)
    )
    natural_parts = sorted(nx.connected_components(graph), key=lambda part: (-len(part), sorted(part)))
    if len(natural_parts) != len(component_records) or len(component_records) != len(component_results):
        raise ValueError("Saved graph component count does not match the reconstruction metadata.")

    producer_summary = pd.read_csv(run_dir / "final_summary.csv").set_index("metric").value
    exact_total = float(producer_summary["Exact components objective sum"])
    component_file = run_dir / "component_mwis_summary.csv"
    component_bounds: dict[str, float] = {}
    component_exact: dict[str, bool] = {}
    if component_file.exists():
        saved = pd.read_csv(component_file).set_index("component_id")
        if set(saved.index.astype(str)) != {str(row["component_id"]) for row in component_records}:
            raise ValueError("Per-component MWIS summary does not match reconstruction components.")
        component_bounds = {str(cid): float(row["upper_bound"]) for cid, row in saved.iterrows()}
        component_exact = {str(cid): bool(row["exact"]) for cid, row in saved.iterrows()}
        exact_sum = sum(
            float(saved.loc[cid, "objective"])
            for cid in saved.index.astype(str)
            if component_exact[cid]
        )
        if abs(exact_sum - exact_total) > TOLERANCE:
            raise ValueError("Per-component exact objectives do not match the producer's exact total.")
    else:
        # Older runs lack per-component bounds. Use the saved classical solution
        # only when its completed-component total matches the certified sum.
        baseline = pd.read_csv(run_dir / "mwis_solution.csv")
        baseline_ids = set(baseline["slot_id"].astype(str))
        complete_total = 0.0
        for record, part in zip(component_records, natural_parts):
            if record.get("status") == "complete":
                complete_total += sum(float(graph.nodes[node]["weight"]) for node in (baseline_ids & part))
        legacy_complete_exact = abs(complete_total - exact_total) <= TOLERANCE
        for record, part in zip(component_records, natural_parts):
            cid = str(record["component_id"])
            baseline_score = sum(float(graph.nodes[node]["weight"]) for node in (baseline_ids & part))
            is_complete_exact = record.get("status") == "complete" and legacy_complete_exact
            component_exact[cid] = is_complete_exact
            component_bounds[cid] = (
                baseline_score if is_complete_exact
                else sum(max(0.0, float(graph.nodes[node]["weight"])) for node in part)
            )

    by_component = {str(row["component_id"]): row for row in component_rows}
    milp_rows: list[dict[str, Any]] = []
    milp_remaining = max(0.0, float(milp_total_seconds))
    lp_remaining = max(0.0, float(lp_total_seconds))
    total_milp_seconds = 0.0
    total_lp_seconds = 0.0
    incomplete_count = 0
    optimal_count = 0

    for index, (record, part) in enumerate(zip(component_records, natural_parts)):
        cid = str(record["component_id"])
        current = dict(component_results[index])
        selected = set(current["selected_nodes"])
        part = set(part)
        if not selected <= part:
            raise ValueError(f"Reconstructed selection contains nodes outside {cid}.")
        previous_bound = float(component_bounds[cid])
        if previous_bound + TOLERANCE < float(current["objective"]):
            raise ValueError(f"Saved upper bound for {cid} is below its feasible schedule.")
        if record.get("status") != "incomplete":
            current["upper_bound"] = previous_bound
            component_results[index] = current
            row = by_component[cid]
            row.update({
                "solution_source": "annealing reconstruction",
                "component_upper_bound": previous_bound,
                "exact_after_hybrid": abs(previous_bound - float(current["objective"])) <= TOLERANCE,
                "milp_optimality_proved": False,
            })
            continue

        incomplete_count += 1
        component_graph = graph.subgraph(part).copy()
        milp_cap = min(float(milp_per_component_seconds), milp_remaining)
        lp_cap = min(float(lp_per_component_seconds), lp_remaining)
        outcome = solve_incomplete_component(
            component_graph,
            selected,
            previous_bound,
            milp_cap,
            lp_cap,
        )
        milp_report = outcome["milp"]
        lp_report = outcome["triangle_lp"]
        if milp_report:
            elapsed = float(milp_report["solve_seconds"])
            total_milp_seconds += elapsed
            milp_remaining = max(0.0, milp_remaining - elapsed)
        if lp_report:
            elapsed = float(lp_report["solve_seconds"])
            total_lp_seconds += elapsed
            lp_remaining = max(0.0, lp_remaining - elapsed)
        if outcome["optimality_proved"]:
            optimal_count += 1

        source = outcome["solution_source"]
        if source.startswith("MILP"):
            current["terminal_ids"] = set()
            current["heuristic_details"] = []
        if outcome["optimality_proved"]:
            current.update({
                "status": "optimal",
                "selected_nodes": set(outcome["selected_nodes"]),
                "heuristic_ids": set(),
                "heuristic_objective": 0.0,
                "heuristic_upper_bound": 0.0,
            })
        else:
            current.update({
                "status": "heuristic_feasible",
                "selected_nodes": set(outcome["selected_nodes"]),
                "heuristic_ids": {cid},
                "heuristic_objective": float(outcome["objective"]),
                "heuristic_upper_bound": float(outcome["upper_bound"]),
            })
            if source.startswith("MILP"):
                current["heuristic_details"] = [{
                    "node_id": cid,
                    "component_id": cid,
                    "reason": "MILP time limit reached; using its feasible incumbent and best valid bound",
                    "node_count": component_graph.number_of_nodes(),
                    "edge_count": component_graph.number_of_edges(),
                    "selected_contacts": len(current["selected_nodes"]),
                    "objective": float(outcome["objective"]),
                    "upper_bound": float(outcome["upper_bound"]),
                    "gap_bound": max(0.0, float(outcome["upper_bound"]) - float(outcome["objective"])),
                    "selected_nodes": json.dumps(sorted(current["selected_nodes"])),
                }]
        current["objective"] = float(outcome["objective"])
        current["upper_bound"] = float(outcome["upper_bound"])
        component_results[index] = current

        row = by_component[cid]
        row.update({
            "status": current["status"],
            "selected_contacts": len(current["selected_nodes"]),
            "objective": current["objective"],
            "terminal_subgraphs_used": len(current["terminal_ids"]),
            "heuristic_fallback_nodes": len(current["heuristic_ids"]),
            "heuristic_objective": current["heuristic_objective"],
            "heuristic_upper_bound": current["heuristic_upper_bound"],
            "solution_source": source,
            "component_upper_bound": current["upper_bound"],
            "exact_after_hybrid": outcome["optimality_proved"],
            "milp_optimality_proved": outcome["optimality_proved"],
        })
        milp_rows.append({
            "component_id": cid,
            "nodes": component_graph.number_of_nodes(),
            "edges": component_graph.number_of_edges(),
            "annealed_objective": round(float(outcome["current_objective"]), 6),
            "milp_objective": milp_report["objective"] if milp_report else None,
            "final_feasible_objective": round(float(outcome["objective"]), 6),
            "previous_upper_bound": round(previous_bound, 6),
            "triangle_lp_upper_bound": lp_report["upper_bound"] if lp_report else None,
            "final_component_upper_bound": round(float(outcome["upper_bound"]), 6),
            "solution_source": source,
            "bound_source": outcome["bound_source"],
            "milp_status": milp_report["status"] if milp_report else "cumulative budget exhausted",
            "milp_optimality_proved": outcome["optimality_proved"],
            "milp_solver_seconds": milp_report["solve_seconds"] if milp_report else 0.0,
            "triangle_lp_status": lp_report["status"] if lp_report else "not required or budget exhausted",
            "triangle_lp_seconds": lp_report["solve_seconds"] if lp_report else 0.0,
            "improvement_vs_annealing": round(float(outcome["objective_improvement"]), 6),
        })

    upper_sum = sum(float(result["upper_bound"]) for result in component_results)
    hybrid_upper = min(float(total_upper_bound), upper_sum)
    hybrid_objective = sum(float(result["objective"]) for result in component_results)
    if hybrid_upper + TOLERANCE < hybrid_objective:
        raise ValueError("Hybrid upper bound is below its feasible objective.")
    selected_all = set().union(*(result["selected_nodes"] for result in component_results))
    if any(left in selected_all and right in selected_all for left, right in graph.edges):
        raise ValueError("The final hybrid schedule contains a conflict.")

    return {
        "component_results": component_results,
        "component_rows": component_rows,
        "milp_rows": milp_rows,
        "incomplete_components": incomplete_count,
        "milp_optimal_components": optimal_count,
        "milp_unresolved_components": incomplete_count - optimal_count,
        "milp_solver_seconds": total_milp_seconds,
        "triangle_lp_seconds": total_lp_seconds,
        "hybrid_upper_bound": hybrid_upper,
        "hybrid_objective": hybrid_objective,
        "hybrid_gap": max(0.0, hybrid_upper - hybrid_objective),
        "hybrid_exact": hybrid_upper <= hybrid_objective + TOLERANCE,
        "wall_seconds": time.perf_counter() - started,
    }


## Reconstruct the MWIS schedule

This section walks the saved decomposition tree, combines separator branches and independent pieces, and includes forced selections. Any heuristic fallback remains explicitly identified in the output. The result is one experiment-level schedule and summary.

If a natural component has an incomplete separator decomposition, the notebook runs a bounded binary MWIS MILP on that whole component after annealing. A proven optimum replaces that component’s annealed/fallback contribution; a valid better incumbent may improve its feasible score. If the MILP is not proven optimal, a triangle-LP bound is used when available, otherwise the saved conservative component bound is retained. Other components keep their annealed reconstruction.

In [ ]:
# Prefer the compressed artifact; keep support for older runs with plain JSON.
RECONSTRUCTION_METADATA_GZIP = INPUT_RUN_DIR / "reconstruction_metadata.json.gz"
RECONSTRUCTION_METADATA_JSON = INPUT_RUN_DIR / "reconstruction_metadata.json"
if RECONSTRUCTION_METADATA_GZIP.exists():
    with gzip.open(RECONSTRUCTION_METADATA_GZIP, "rt", encoding="utf-8") as metadata_file:
        reconstruction_metadata = json.load(metadata_file)
elif RECONSTRUCTION_METADATA_JSON.exists():
    reconstruction_metadata = json.loads(RECONSTRUCTION_METADATA_JSON.read_text(encoding="utf-8"))
else:
    raise FileNotFoundError(
        f"No reconstruction_metadata.json[.gz] in {INPUT_RUN_DIR}. "
        "Run the MWIS pipeline first, then point qubo_input_run to that MWIS run."
    )
if reconstruction_metadata.get("producer") != "mwis_pipeline":
    raise ValueError("This reconstruction section currently supports MWIS producer runs only.")

producer_summary_path = INPUT_RUN_DIR / "final_summary.csv"
if not producer_summary_path.exists():
    raise FileNotFoundError(f"MWIS producer summary not found: {producer_summary_path}")
producer_summary = pd.read_csv(producer_summary_path)
producer_summary_values = dict(zip(producer_summary["metric"], producer_summary["value"]))
producer_upper_bound = float(producer_summary_values["Total conservative upper bound"])
producer_decomposition = pd.read_csv(INPUT_RUN_DIR / "decomposition_summary.csv")
incomplete_node_count = int(producer_decomposition["incomplete_nodes"].fillna(0).astype(int).sum())
incomplete_component_count = int((producer_decomposition["incomplete_nodes"].fillna(0).astype(int) > 0).sum())
decomposition_by_component = {
    row["component_id"]: {
        "status": row["status"],
        "incomplete_nodes": int(row["incomplete_nodes"]) if pd.notna(row["incomplete_nodes"]) else 0,
    }
    for row in producer_decomposition.to_dict(orient="records")
}

node_by_id = {node["node_id"]: node for node in reconstruction_metadata["nodes"]}
terminal_by_id = {terminal["terminal_id"]: terminal for terminal in terminal_subgraphs}
annealing_by_terminal = {row["terminal_id"]: row for row in annealing_rows}


def selected_from_row(row: dict[str, Any]) -> set[str]:
    return set(json.loads(row["selected_nodes"]))


def combine_status(statuses: list[str]) -> str:
    if any(status == "annealed_infeasible" for status in statuses):
        return "annealed_infeasible"
    if any(status == "heuristic_feasible" for status in statuses):
        return "annealed_with_heuristic_fallback"
    return "annealed_feasible"


def evaluate_reconstruction_node(node_id: str, memo: dict[str, dict[str, Any]] | None = None) -> dict[str, Any]:
    """Rebuild a feasible schedule by combining saved terminal and fallback choices.

    Memoization avoids evaluating the same shared decomposition node twice.
    """
    if memo is None:
        memo = {}
    if node_id in memo:
        return memo[node_id]

    node = node_by_id[node_id]
    base_selected = set(node.get("forced_selected", []))
    base_objective = float(node.get("forced_objective", 0.0))

    if node["node_type"] == "terminal":
        terminal_id = node.get("terminal_id")
        if terminal_id is None:
            result = {
                "status": "annealed_feasible",
                "selected_nodes": base_selected,
                "objective": base_objective,
                "terminal_ids": set(),
                "heuristic_ids": set(),
                "heuristic_objective": 0.0,
                "heuristic_upper_bound": 0.0,
                "heuristic_details": [],
            }
        else:
            row = annealing_by_terminal[terminal_id]
            terminal_record = terminal_by_id[terminal_id]
            forced_selected = set(terminal_record.get("forced_selected", []))
            selected = forced_selected | selected_from_row(row)
            result = {
                "status": "annealed_feasible" if row["is_independent"] else "annealed_infeasible",
                "selected_nodes": selected,
                "objective": float(terminal_record.get("forced_objective", 0.0)) + float(row["objective"]),
                "terminal_ids": {terminal_id},
                "heuristic_ids": set(),
                "heuristic_objective": 0.0,
                "heuristic_upper_bound": 0.0,
                "heuristic_details": [],
            }
        memo[node_id] = result
        return result

    if node["node_type"] == "incomplete":
        heuristic = node["heuristic_solution"]
        detail = {
            "node_id": node_id,
            "component_id": node["component_id"],
            "reason": node.get("reason", "incomplete decomposition"),
            "node_count": node["qpu_node_count"],
            "edge_count": node["qpu_edge_count"],
            "selected_contacts": len(heuristic["selected_nodes"]),
            "objective": float(heuristic["objective"]),
            "upper_bound": float(heuristic["upper_bound"]),
            "gap_bound": float(heuristic["upper_bound"]) - float(heuristic["objective"]),
            "selected_nodes": json.dumps(sorted(heuristic["selected_nodes"])),
        }
        result = {
            "status": "heuristic_feasible",
            "selected_nodes": set(heuristic["selected_nodes"]),
            "objective": float(heuristic["objective"]),
            "terminal_ids": set(),
            "heuristic_ids": {node_id},
            "heuristic_objective": float(heuristic["objective"]),
            "heuristic_upper_bound": float(heuristic["upper_bound"]),
            "heuristic_details": [detail],
        }
        memo[node_id] = result
        return result

    if node["node_type"] == "independent_sum":
        children = [evaluate_reconstruction_node(child_id, memo) for child_id in node["children"]]
        result = {
            "status": combine_status([child["status"] for child in children]),
            "selected_nodes": base_selected | set().union(*(child["selected_nodes"] for child in children)),
            "objective": base_objective + sum(child["objective"] for child in children),
            "terminal_ids": set().union(*(child["terminal_ids"] for child in children)),
            "heuristic_ids": set().union(*(child["heuristic_ids"] for child in children)),
            "heuristic_objective": sum(child.get("heuristic_objective", 0.0) for child in children),
            "heuristic_upper_bound": sum(child.get("heuristic_upper_bound", 0.0) for child in children),
            "heuristic_details": [detail for child in children for detail in child.get("heuristic_details", [])],
        }
        memo[node_id] = result
        return result

    branch_results = []
    for branch in node["branches"]:
        children = [evaluate_reconstruction_node(child_id, memo) for child_id in branch["children"]]
        selected = base_selected | set(branch["selected_separator"])
        for child in children:
            selected.update(child["selected_nodes"])
        branch_results.append({
            "status": combine_status([child["status"] for child in children]),
            "selected_nodes": selected,
            "objective": base_objective + float(branch["state_objective"]) + sum(child["objective"] for child in children),
            "terminal_ids": set().union(*(child["terminal_ids"] for child in children)) if children else set(),
            "heuristic_ids": set().union(*(child["heuristic_ids"] for child in children)) if children else set(),
            "heuristic_objective": sum(child.get("heuristic_objective", 0.0) for child in children),
            "heuristic_upper_bound": sum(child.get("heuristic_upper_bound", 0.0) for child in children),
            "heuristic_details": [detail for child in children for detail in child.get("heuristic_details", [])],
            "branch_state_index": branch["state_index"],
        })

    best = max(branch_results, key=lambda item: (item["objective"], tuple(sorted(item["selected_nodes"]))))
    memo[node_id] = best
    return best


component_rows = []
component_results = []
reconstruction_memo: dict[str, dict[str, Any]] = {}
for component in reconstruction_metadata["components"]:
    result = evaluate_reconstruction_node(component["root_node_id"], reconstruction_memo)
    component_results.append(result)
    decomposition = decomposition_by_component.get(component["component_id"], {})
    component_rows.append({
        "component_id": component["component_id"],
        "status": result["status"],
        "selected_contacts": len(result["selected_nodes"]),
        "objective": result["objective"],
        "terminal_subgraphs_used": len(result["terminal_ids"]),
        "heuristic_fallback_nodes": len(result["heuristic_ids"]),
        "heuristic_objective": result.get("heuristic_objective", 0.0),
        "heuristic_upper_bound": result.get("heuristic_upper_bound", 0.0),
        "decomposition_status": decomposition.get("status", "unknown"),
        "incomplete_decomposition_nodes_all_states": decomposition.get("incomplete_nodes", 0),
    })

# Replace only separator-incomplete natural components; all other schedules remain annealed.
annealed_objective_before_milp = sum(float(result["objective"]) for result in component_results)
hybrid_run = apply_milp_fallback(
    INPUT_RUN_DIR,
    reconstruction_metadata["components"],
    component_results,
    component_rows,
    producer_upper_bound,
    milp_per_component_seconds=MILP_PER_COMPONENT_SECONDS,
    milp_total_seconds=MILP_TOTAL_SOLVER_SECONDS,
    lp_per_component_seconds=TRIANGLE_LP_PER_COMPONENT_SECONDS,
    lp_total_seconds=TRIANGLE_LP_TOTAL_SOLVER_SECONDS,
)
component_results = hybrid_run["component_results"]
component_rows = hybrid_run["component_rows"]
milp_component_rows = hybrid_run["milp_rows"]

selected_nodes = set().union(*(result["selected_nodes"] for result in component_results))
total_objective = sum(result["objective"] for result in component_results)
heuristic_ids = set().union(*(result["heuristic_ids"] for result in component_results))
terminal_ids_used = set().union(*(result["terminal_ids"] for result in component_results))
infeasible_terminals = [row for row in annealing_rows if not row["is_independent"]]
heuristic_details = [detail for result in component_results for detail in result.get("heuristic_details", [])]
heuristic_objective = sum(result.get("heuristic_objective", 0.0) for result in component_results)
heuristic_upper_bound = sum(result.get("heuristic_upper_bound", 0.0) for result in component_results)
heuristic_gap_bound = max(0.0, heuristic_upper_bound - heuristic_objective)
hybrid_upper_bound = float(hybrid_run["hybrid_upper_bound"])
hybrid_gap_bound = float(hybrid_run["hybrid_gap"])
hybrid_relative_gap_percent = 0.0 if hybrid_upper_bound == 0 else 100 * hybrid_gap_bound / hybrid_upper_bound
annealed_gap_bound = producer_upper_bound - annealed_objective_before_milp
if annealed_gap_bound < -1e-6:
    raise ValueError("Annealed objective exceeds the MWIS producer upper bound; check input/run matching.")
annealed_gap_bound = max(0.0, annealed_gap_bound)
annealed_relative_gap_percent = 0.0 if producer_upper_bound == 0 else 100 * annealed_gap_bound / producer_upper_bound

producer_config_path = INPUT_RUN_DIR / "run_config_log.json"
producer_config = json.loads(producer_config_path.read_text(encoding="utf-8")) if producer_config_path.exists() else run_config
slot_csv = PROJECT_ROOT / producer_config["candidate_slot_csv"]
slot_df = pd.read_csv(slot_csv)
reconstructed_schedule = slot_df[slot_df["slot_id"].isin(selected_nodes)].copy()
if len(reconstructed_schedule) != len(selected_nodes):
    raise ValueError("Reconstructed schedule is missing or duplicates selected candidate slots.")
schedule_objective = float(reconstructed_schedule["weight"].sum())
if abs(schedule_objective - total_objective) > 1e-6:
    raise ValueError("Reconstructed schedule objective does not match the selected-node objective.")

component_provenance = pd.DataFrame(component_rows)
heuristic_fallback_details = pd.DataFrame(
    heuristic_details,
    columns=[
        "node_id",
        "component_id",
        "reason",
        "node_count",
        "edge_count",
        "selected_contacts",
        "objective",
        "upper_bound",
        "gap_bound",
        "selected_nodes",
    ],
)
summary_rows = [
    ("Producer run", INPUT_RUN_DIR.name),
    ("Status", "hybrid_exact" if hybrid_run["hybrid_exact"] else "hybrid_feasible_not_certified"),
    ("Selected contacts", len(selected_nodes)),
    ("Total feasible objective", total_objective),
    ("Annealed terminal subgraphs available", len(annealing_rows)),
    ("Annealed terminal subgraphs used", len(terminal_ids_used)),
    ("Infeasible terminal annealing samples", len(infeasible_terminals)),
    ("Fallback nodes/components used in selected schedule", len(heuristic_ids)),
    ("Selected-schedule fallback feasible objective", heuristic_objective),
    ("Selected-schedule fallback upper bound", heuristic_upper_bound),
    ("Selected-schedule fallback gap bound", heuristic_gap_bound),
    ("Components with incomplete separator decomposition (all states)", incomplete_component_count),
    ("Incomplete separator nodes (all states)", incomplete_node_count),
    ("MWIS producer conservative objective upper bound", producer_upper_bound),
    ("Hybrid conservative objective upper bound", hybrid_upper_bound),
    ("Hybrid schedule absolute gap bound", hybrid_gap_bound),
    ("Hybrid schedule relative gap bound (%)", hybrid_relative_gap_percent),
    ("MILP incomplete components attempted", hybrid_run["incomplete_components"]),
    ("MILP components proven optimal", hybrid_run["milp_optimal_components"]),
    ("MILP components unresolved", hybrid_run["milp_unresolved_components"]),
    ("MILP solver time (seconds)", hybrid_run["milp_solver_seconds"]),
    ("Triangle-LP fallback solver time (seconds)", hybrid_run["triangle_lp_seconds"]),
    ("Original annealed schedule absolute gap bound", annealed_gap_bound),
    ("Original annealed schedule relative gap bound (%)", annealed_relative_gap_percent),
    ("Exact global objective", total_objective if hybrid_run["hybrid_exact"] else "not certified"),
    ("Certification", "hybrid schedule matches the summed component upper bounds" if hybrid_run["hybrid_exact"] else "feasible hybrid schedule; optimality not certified"),
]
reconstructed_summary = pd.DataFrame(summary_rows, columns=["metric", "value"])

reconstructed_schedule.to_csv(OUTPUT_RUN_DIR / "reconstructed_schedule.csv", index=False)
pd.DataFrame(milp_component_rows).to_csv(OUTPUT_RUN_DIR / "milp_component_results.csv", index=False)
component_provenance.to_csv(OUTPUT_RUN_DIR / "component_provenance_summary.csv", index=False)
heuristic_fallback_details.to_csv(OUTPUT_RUN_DIR / "heuristic_fallback_details.csv", index=False)
reconstructed_summary.to_csv(OUTPUT_RUN_DIR / "reconstructed_summary.csv", index=False)
(OUTPUT_RUN_DIR / "reconstructed_selected_nodes.json").write_text(
    json.dumps({"selected_nodes": sorted(selected_nodes)}, indent=2),
    encoding="utf-8",
)

print({
    "selected_contacts": len(selected_nodes),
    "total_objective": total_objective,
    "infeasible_terminal_samples": len(infeasible_terminals),
    "heuristic_fallback_nodes": len(heuristic_ids),
    "heuristic_gap_bound": heuristic_gap_bound,
    "output": str(OUTPUT_RUN_DIR),
})
reconstructed_summary